# 动手练习：同步与协同动力学

来自《生物医学网络科学》（Biomedical Network Science）。 参考答案版。

In [ ]:
# 数据文件（JupyterLite 专用；文件已在时什么也不做）
import os, js
from pyodide.http import pyfetch
_base = str(js.location.href).split("/extensions/")[0].split("/build/")[0] + "/files/practice/data/"
os.makedirs("data", exist_ok=True)
for _f in ['lausanne114_nodes.csv', 'lausanne114_sc_edges.csv']:
    if not os.path.exists("data/" + _f):
        _r = await pyfetch(_base + _f)
        open("data/" + _f, "wb").write(await _r.bytes())


## 任务 1：序参量与洛伦兹分布下的理论曲线

Kuramoto 序参量 $r e^{i\psi}=\frac1N\sum_j e^{i\theta_j}$ 把每个振荡器看作单位圆上的一点，$r$ 是这些点“质心”的模长。

1. 写函数 `order_parameter(theta)`，输入相位数组，返回 $r$（一个 0 到 1 之间的实数）；
2. 写函数 `r_theory(K, gamma)`：固有频率服从半宽为 $\gamma$ 的洛伦兹分布时，全连接模型的稳态序参量。$K\le K_c=2\gamma$ 时为 0，$K>K_c$ 时为 $\sqrt{1-2\gamma/K}$。

期望看到：相位 $0,\pi/2,\pi,3\pi/2$ 给出 $r=0$；相位 $0,0.1,-0.1,0$ 给出 $r\approx0.998$；$\gamma=0.5$ 时 $r(K{=}2)\approx0.71$、$r(K{=}4)\approx0.87$。

> **提示：** `np.abs(np.exp(1j * theta).mean())`；注意不要对 `cos` 单独求平均，那只是质心的横坐标。

In [ ]:
# 准备（直接运行）
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

In [ ]:
def order_parameter(theta):
    return float(np.abs(np.exp(1j * np.asarray(theta)).mean()))

def r_theory(K, gamma):
    return float(np.sqrt(1 - 2 * gamma / K)) if K > 2 * gamma else 0.0

print("r(0, π/2, π, 3π/2) = %.3f" % order_parameter([0, np.pi / 2, np.pi, 3 * np.pi / 2]))
print("r(0, 0.1, -0.1, 0) = %.3f" % order_parameter([0, 0.1, -0.1, 0]))
print("理论值 γ=0.5：K=2 → %.3f，K=4 → %.3f" % (r_theory(2, 0.5), r_theory(4, 0.5)))

In [ ]:
# 核对（不用修改）
assert order_parameter([0.0, 1.0]) is not None, "order_parameter 还没有返回值"
assert abs(order_parameter([0, np.pi / 2, np.pi, 3 * np.pi / 2])) < 1e-9, "四个均匀分布的相位应给出 r=0；检查是否对 exp(1j*theta) 求平均后再取模"
assert abs(order_parameter([0, 0.1, -0.1, 0]) - 0.998) < 0.002, "相位 0, 0.1, -0.1, 0 的 r 应约为 0.998"
assert abs(order_parameter([0, np.pi]) - 0) < 1e-9, "两个反相的相位 r 应为 0；若得到 1，可能是先取模再平均了"
assert abs(order_parameter([0, 0, np.pi / 2]) - np.sqrt(5) / 3) < 1e-6, "相位 0, 0, π/2 的 r 应为 √5/3 ≈ 0.745；只平均 cos 会得到 0.667"
assert r_theory(0.5, 0.5) == 0, "K < K_c = 2γ 时只有不相干解，r 应为 0"
assert abs(r_theory(2, 0.5) - 0.707) < 0.002, "γ=0.5、K=2 时 r = √(1-1/2) ≈ 0.71"
assert abs(r_theory(4, 0.5) - 0.866) < 0.002, "γ=0.5、K=4 时 r = √0.75 ≈ 0.87；检查 K_c 是否用了 2γ"
print("✓ 任务 1 通过")

## 任务 2：全连接 Kuramoto 模型的同步相变

固有频率取洛伦兹分布（半宽 $\gamma=0.5$，理论 $K_c=1$）。下面的 `simulate` 用平均场形式 $\dot\theta_i=\omega_i+Kr\sin(\psi-\theta_i)$ 做欧拉积分，返回后一半时间的平均 $r$。

1. 对 `Ks` 中每个 $K$ 调用 `simulate`，得到数组 `r_sim`；
2. 用任务 1 的 `r_theory` 得到对应的理论值数组 `r_th`；
3. 把两者画在一张图上。

期望看到：$K<1$ 时 `r_sim` 只有约 0.06（$N=200$ 的有限尺度基线，量级 $1/\sqrt N$），$K>1$ 后迅速上升并贴近理论曲线（$K=2$ 约 0.71，$K=3$ 约 0.82）。

> **提示：** `r_sim = np.array([simulate(K, omega) for K in Ks])`。

In [ ]:
# 准备（直接运行）
N, gamma = 200, 0.5
# 用分位数取样代替随机抽样，减少有限尺度涨落（洛伦兹分布的反函数为 gamma*tan）
omega = gamma * np.tan(np.pi * (np.arange(1, N + 1) / (N + 1) - 0.5))

def simulate(K, omega, T=60.0, dt=0.05, seed=0):
    theta = np.random.default_rng(seed).uniform(0, 2 * np.pi, omega.size)
    rs = []
    for _ in range(int(T / dt)):
        z = np.exp(1j * theta).mean()            # 序参量 r e^{i psi}
        theta = theta + dt * (omega + K * np.imag(z * np.exp(-1j * theta)))
        rs.append(abs(z))
    return np.mean(rs[len(rs) // 2:])

Ks = np.arange(0, 3.01, 0.25)

In [ ]:
r_sim = np.array([simulate(K, omega) for K in Ks])
r_th = np.array([r_theory(K, gamma) for K in Ks])
plt.figure(figsize=(5, 3.5))
plt.plot(Ks, r_th, "k-", label="theory")
plt.plot(Ks, r_sim, "o", label="simulation")
plt.axvline(2 * gamma, ls="--", c="gray")
plt.xlabel("K"); plt.ylabel("r"); plt.legend(); plt.title("Kuramoto phase transition")
for K, a, b in zip(Ks, r_sim, r_th):
    print("K = %.2f   模拟 r = %.3f   理论 r = %.3f" % (K, a, b))

In [ ]:
# 核对（不用修改）
assert r_sim is not None and r_th is not None, "r_sim 或 r_th 还没有赋值"
r_sim, r_th = np.asarray(r_sim, float), np.asarray(r_th, float)
assert r_sim.shape == Ks.shape and r_th.shape == Ks.shape, "r_sim、r_th 的长度应与 Ks 相同（每个 K 一个值）"
assert np.all(r_sim[Ks < 0.8] < 0.15), "K < K_c 时模拟的 r 应只有有限尺度基线（约 0.06）；检查是否传入了正确的 K"
assert abs(r_sim[Ks == 2.0][0] - 0.71) < 0.06, "K=2 时模拟 r 应约为 0.71；检查是否取了后一半时间的平均"
assert abs(r_sim[Ks == 3.0][0] - 0.82) < 0.06, "K=3 时模拟 r 应约为 0.82"
assert np.allclose(r_th[Ks > 1.2], np.sqrt(1 - 2 * gamma / Ks[Ks > 1.2]), atol=1e-3), "理论值应为 √(1-2γ/K)"
assert np.all(np.abs(r_sim - r_th)[Ks >= 1.5] < 0.06), "K ≥ 1.5 时模拟值应与理论值相差不超过约 0.05"
print("✓ 任务 2 通过")

## 任务 3：网络结构如何影响同步起点

在 ER 与 BA 网络（$N=300$，边数相同）上运行网络 Kuramoto 模型 $\dot\theta_i=\omega_i+K\sum_jA_{ij}\sin(\theta_j-\theta_i)$，固有频率取标准高斯分布，$K_0=2/(\pi g(0))=2\sqrt{2\pi}/\pi\approx1.60$。

1. 写函数 `Kc_hmf(G)`，返回异质平均场估计 $K_c=K_0\langle k\rangle/\langle k^2\rangle$；
2. 用 `net_kuramoto` 在 `Ks2` 上分别得到 `r_er` 与 `r_ba` 两个数组。

期望看到：BA 网络 $K_c\approx0.14$，ER 网络 $K_c\approx0.23$；BA 的 $r(K)$ 更早上升，$K$ 较大时两者接近。

> **提示：** $\langle k^2\rangle$ 是 `(k**2).mean()`，不是 `k.mean()**2`。

In [ ]:
# 准备（直接运行）
import scipy.sparse as sp

N2 = 300
G_ba = nx.barabasi_albert_graph(N2, 3, seed=2)
G_er = nx.gnm_random_graph(N2, G_ba.number_of_edges(), seed=2)
omega2 = np.random.default_rng(5).standard_normal(N2)
K0 = 2 * np.sqrt(2 * np.pi) / np.pi

def net_kuramoto(G, K, omega, T=40.0, dt=0.05, seed=0):
    A = sp.csr_matrix(nx.to_scipy_sparse_array(G, nodelist=range(G.number_of_nodes())))
    theta = np.random.default_rng(seed).uniform(0, 2 * np.pi, omega.size)
    rs = []
    for _ in range(int(T / dt)):
        s, c = np.sin(theta), np.cos(theta)
        coupling = c * (A @ s) - s * (A @ c)       # = sum_j A_ij sin(theta_j - theta_i)
        theta = theta + dt * (omega + K * coupling)
        rs.append(abs(np.exp(1j * theta).mean()))
    return np.mean(rs[len(rs) // 2:])

Ks2 = np.round(np.arange(0, 0.61, 0.1), 2)

In [ ]:
def Kc_hmf(G):
    k = np.array([d for _, d in G.degree()], dtype=float)
    return K0 * k.mean() / (k ** 2).mean()

r_er = np.array([net_kuramoto(G_er, K, omega2) for K in Ks2])
r_ba = np.array([net_kuramoto(G_ba, K, omega2) for K in Ks2])
print("K_c 估计：ER = %.3f，BA = %.3f" % (Kc_hmf(G_er), Kc_hmf(G_ba)))
plt.figure(figsize=(5, 3.5))
plt.plot(Ks2, r_er, "o-", label="ER"); plt.plot(Ks2, r_ba, "s-", label="BA")
plt.xlabel("K"); plt.ylabel("r"); plt.legend(); plt.title("Kuramoto on networks")
print("ER:", np.round(r_er, 2).tolist()); print("BA:", np.round(r_ba, 2).tolist())

In [ ]:
# 核对（不用修改）
assert Kc_hmf(G_ba) is not None, "Kc_hmf 还没有返回值"
assert abs(Kc_hmf(G_ba) - 0.14) < 0.02, "BA 网络的 K_c 估计应约为 0.14；检查 <k^2> 是否写成了 <k>^2"
assert abs(Kc_hmf(G_er) - 0.23) < 0.02, "ER 网络的 K_c 估计应约为 0.23"
assert r_er is not None and r_ba is not None, "r_er 或 r_ba 还没有赋值"
r_er, r_ba = np.asarray(r_er, float), np.asarray(r_ba, float)
assert r_er.shape == Ks2.shape and r_ba.shape == Ks2.shape, "r_er、r_ba 的长度应与 Ks2 相同"
assert r_ba[Ks2 == 0.2][0] > r_er[Ks2 == 0.2][0] + 0.05, "K=0.2 时 BA 网络的 r 应明显高于 ER（枢纽带动同步更早开始）"
assert r_ba[-1] > 0.6 and r_er[-1] > 0.6, "K=0.6 时两种网络都应已明显同步（r > 0.6）"
print("✓ 任务 3 通过")

## 任务 4：模块化网络上的分层同步与拉普拉斯谱

4 个社团、每个 40 个节点，社团内连接概率 0.3，社团间 0.005。所有振荡器固有频率相同（=0），初始相位随机。

1. 在给定的积分循环中，每一步记录全局序参量（追加到 `r_global`）和 4 个社团各自序参量的平均（追加到 `r_local`）；
2. 计算拉普拉斯矩阵的全部特征值，从小到大存为 `mu`；
3. 在前 10 个特征值中找相邻两个比值 `mu[i+1]/mu[i]`（$i\ge1$）最大的位置，`n_slow` = 这个间隙之前的特征值个数（含 $\mu_1=0$）。

期望看到：$t=20$（第 200 步）时社团内 $r\approx0.98$，全局 $r$ 只有约 0.46；拉普拉斯谱有 4 个很小的特征值（含 0），之后出现明显间隙——社团之间的慢模式对应全局同步的慢过程。

> **提示：** `abs(z[labels == c].mean())` 是社团 c 的序参量；`np.sort(np.linalg.eigvalsh(L))`。

In [ ]:
# 准备（直接运行）
sizes = [40, 40, 40, 40]
G4 = nx.random_partition_graph(sizes, 0.3, 0.005, seed=3)
A4 = sp.csr_matrix(nx.to_scipy_sparse_array(G4, nodelist=range(G4.number_of_nodes())))
labels = np.repeat(np.arange(len(sizes)), sizes)
K4, dt4, steps4 = 0.05, 0.1, 1500

In [ ]:
theta = np.random.default_rng(1).uniform(0, 2 * np.pi, G4.number_of_nodes())
r_local, r_global = [], []
for _ in range(steps4):
    s, c = np.sin(theta), np.cos(theta)
    theta = theta + dt4 * K4 * (c * (A4 @ s) - s * (A4 @ c))
    z = np.exp(1j * theta)
    r_global.append(abs(z.mean()))
    r_local.append(np.mean([abs(z[labels == q].mean()) for q in range(len(sizes))]))

L4 = nx.laplacian_matrix(G4, nodelist=range(G4.number_of_nodes())).toarray().astype(float)
mu = np.sort(np.linalg.eigvalsh(L4))
ratios = mu[2:10] / mu[1:9]
n_slow = int(np.argmax(ratios)) + 2
t = np.arange(steps4) * dt4
plt.figure(figsize=(5, 3.5))
plt.plot(t, r_local, label="within communities"); plt.plot(t, r_global, label="global")
plt.xlabel("time"); plt.ylabel("r"); plt.legend(); plt.title("Hierarchical synchronization")
print("t=20 时 r_local = %.2f，r_global = %.2f" % (r_local[200], r_global[200]))
print("最小的 8 个拉普拉斯特征值:", np.round(mu[:8], 3).tolist(), " n_slow =", n_slow)

In [ ]:
# 核对（不用修改）
assert len(r_local) == steps4 and len(r_global) == steps4, "r_local 和 r_global 都应在每一步追加一个值（共 %d 个）" % steps4
assert r_local[200] > 0.9, "t=20 时社团内平均 r 应已接近 1（约 0.98）；检查是否对每个社团分别求序参量再平均"
assert r_global[200] < 0.75, "t=20 时全局 r 应明显低于社团内 r（约 0.46）；检查 r_global 是否用了全部节点"
assert r_local[200] - r_global[200] > 0.2, "分层同步：社团内同步应明显快于全局同步"
assert mu is not None and len(mu) == G4.number_of_nodes(), "mu 应包含全部 160 个拉普拉斯特征值"
assert abs(mu[0]) < 1e-8 and np.all(np.diff(mu) >= -1e-9), "mu 应从小到大排列，最小值为 0"
assert n_slow == 4, "4 个弱连接的社团应给出 4 个小特征值（含 0）；检查比值是否从 mu[2]/mu[1] 开始算"
print("✓ 任务 4 通过")

## 任务 5：共识动力学与代数连通度

共识动力学 $\dot{\mathbf x}=-L\mathbf x$ 的解为 $\mathbf x(t)=\bar x\mathbf 1+\sum_{k\ge2}e^{-\mu_kt}c_k\mathbf v_k$，偏离平均值的误差最慢按 $e^{-\mu_2t}$ 衰减。在 4 种网络（$N=60$，平均度 4）上，从相同的随机初值出发：

1. 求每个网络的代数连通度，存入字典 `mu2`（键为网络名）；
2. 用谱分解计算误差 $\|\mathbf x(t)-\bar x\mathbf 1\|$ 首次降到初始值 1% 的时间，存入字典 `t_cons`（时间网格已给出）；
3. 计算每个网络的乘积 $\mu_2\times$ 时间。

期望看到：环 $\mu_2\approx0.055$、用时约 65；重连概率 0.5 的 WS 网络 $\mu_2\approx0.57$、用时约 5；$\mu_2$ 越大收敛越快，$\mu_2\times$时间在各网络间量级相近（约 2–5；若误差只在 $\mathbf v_2$ 上，它等于 $\ln 100\approx4.6$）。

> **提示：** `mu_k, V = np.linalg.eigh(L)`，`c = V.T @ (x0 - x0.mean())`，误差为 `np.sqrt(((np.exp(-np.outer(ts, mu_k)) * c) ** 2).sum(axis=1))`。

In [ ]:
# 准备（直接运行）
N5 = 60
nets = {
    "ring": nx.watts_strogatz_graph(N5, 4, 0.0, seed=1),
    "WS p=0.1": nx.connected_watts_strogatz_graph(N5, 4, 0.1, seed=1),
    "WS p=0.5": nx.connected_watts_strogatz_graph(N5, 4, 0.5, seed=1),
    "complete": nx.complete_graph(N5),
}
x0 = np.random.default_rng(0).standard_normal(N5)
ts = np.arange(0, 200, 0.01)

In [ ]:
mu2, t_cons = {}, {}
for name, G in nets.items():
    L = nx.laplacian_matrix(G, nodelist=range(N5)).toarray().astype(float)
    mu_k, V = np.linalg.eigh(L)
    c = V.T @ (x0 - x0.mean())
    err = np.sqrt(((np.exp(-np.outer(ts, mu_k)) * c) ** 2).sum(axis=1))
    mu2[name] = mu_k[1]
    t_cons[name] = ts[np.argmax(err <= 0.01 * err[0])]
    print("%-10s mu2 = %.4f   收敛时间 = %6.2f   mu2*时间 = %.2f" % (name, mu2[name], t_cons[name], mu2[name] * t_cons[name]))

In [ ]:
# 核对（不用修改）
assert set(mu2) == set(nets) and set(t_cons) == set(nets), "mu2 与 t_cons 应包含全部 4 个网络"
assert abs(mu2["ring"] - 0.055) < 0.005, "环的 mu2 应约为 0.055；检查是否取了第二小的特征值（最小的是 0）"
assert abs(mu2["complete"] - 60) < 1e-6, "完全图 K_N 的 mu2 = N = 60"
assert abs(t_cons["ring"] - 65) < 8, "环的收敛时间应约为 65；检查误差是否相对初始误差 1%"
assert abs(t_cons["WS p=0.5"] - 5) < 1.5, "WS p=0.5 的收敛时间应约为 5"
order_mu = sorted(nets, key=lambda n: mu2[n])
order_t = sorted(nets, key=lambda n: -t_cons[n])
assert order_mu == order_t, "mu2 越大，收敛应越快：两种排序应一致"
assert all(0.5 < mu2[n] * t_cons[n] < 6 for n in nets), "mu2 × 时间应在 ln100 ≈ 4.6 附近或略小"
print("✓ 任务 5 通过")

## 任务 6：开放小分析：人脑结构连接组的可同步性

用 70 名健康成人的组水平结构连接组（Lausanne 图谱 114 个皮层区，Shafiei 等 2019，Zenodo，CC BY 4.0，见 `data/SOURCES.md`），把权重（纤维密度）除以平均强度使 $\langle s\rangle=1$，然后自己设计一个小分析，例如：

- 计算拉普拉斯谱的 $\mu_2$ 与特征值比 $\mu_N/\mu_2$，并与一个保度随机化（再随机分配权重）的对照网络比较；
- 用强度代替度，估计 $K_c\approx K_0\langle s\rangle/\langle s^2\rangle$，与最大特征值近似 $K_0/\Lambda_{\max}$ 比较；
- 在若干个 $K$ 上模拟网络 Kuramoto 模型，得到 $r(K)$ 曲线，看真实网络与对照网络谁先同步。

至少给出以下变量：`mu2_brain`（正数）、`ratio_brain`（$\mu_N/\mu_2$，≥ 1）、`Ks6`（你选的耦合强度数组）和 `r_curve`（与 `Ks6` 等长、取值在 0–1 的数组）。用两三句话写下你的解读：真实连接组的 $\mu_2$ 和特征值比与对照相比如何，可能与什么结构有关（例如半球间连接少）？这对“相同振荡器的完全同步”意味着什么？注意平均场估计只是近似。

> **提示：** `W[edges.i, edges.j] = edges.fiber_density; W = W + W.T`；保度随机化可用 `nx.double_edge_swap`；模拟时把邻接矩阵换成 `W` 即可。

In [ ]:
# 准备（直接运行）
import pandas as pd

edges = pd.read_csv("data/lausanne114_sc_edges.csv")
nodes = pd.read_csv("data/lausanne114_nodes.csv")
N6 = len(nodes)

def kuramoto_W(W, K, omega, T=60.0, dt=0.05, seed=1):
    theta = np.random.default_rng(seed).uniform(0, 2 * np.pi, omega.size)
    rs = []
    for _ in range(int(T / dt)):
        s, c = np.sin(theta), np.cos(theta)
        theta = theta + dt * (omega + K * (c * (W @ s) - s * (W @ c)))
        rs.append(abs(np.exp(1j * theta).mean()))
    return np.mean(rs[len(rs) // 2:])

In [ ]:
W = np.zeros((N6, N6))
W[edges.i, edges.j] = edges.fiber_density
W = W + W.T
W = W / W.sum(axis=1).mean()
s6 = W.sum(axis=1)

def lap_stats(W):
    mu = np.linalg.eigvalsh(np.diag(W.sum(axis=1)) - W)
    return mu[1], mu[-1] / mu[1]

mu2_brain, ratio_brain = lap_stats(W)

# 保度随机化对照：交换边保持每个节点的度，再把原来的权重随机分配给新边
Gb = nx.from_numpy_array((W > 0).astype(int))
Gr = Gb.copy()
nx.double_edge_swap(Gr, nswap=10 * Gb.number_of_edges(), max_tries=10 ** 6, seed=7)
w_old = W[np.triu_indices(N6, 1)]
w_old = w_old[w_old > 0]
W_rand = np.zeros_like(W)
rng6 = np.random.default_rng(7)
for (u, v), w in zip(Gr.edges(), rng6.permutation(w_old)):
    W_rand[u, v] = W_rand[v, u] = w
W_rand = W_rand / W_rand.sum(axis=1).mean()
mu2_rand, ratio_rand = lap_stats(W_rand)
print("真实连接组：mu2 = %.4f，mu_N/mu2 = %.1f" % (mu2_brain, ratio_brain))
print("保度随机化：mu2 = %.4f，mu_N/mu2 = %.1f" % (mu2_rand, ratio_rand))
print("K_c 估计：异质平均场 %.2f，最大特征值近似 %.2f" % (K0 * s6.mean() / (s6 ** 2).mean(), K0 / np.linalg.eigvalsh(W).max()))

omega6 = np.random.default_rng(0).standard_normal(N6)
Ks6 = np.linspace(0, 4, 9)
r_curve = np.array([kuramoto_W(W, K, omega6) for K in Ks6])
r_rand = np.array([kuramoto_W(W_rand, K, omega6) for K in Ks6])
plt.figure(figsize=(5, 3.5))
plt.plot(Ks6, r_curve, "o-", label="brain SC"); plt.plot(Ks6, r_rand, "s--", label="degree-preserving null")
plt.xlabel("K"); plt.ylabel("r"); plt.legend(); plt.title("Kuramoto on the human connectome")
print("真实 r(K):", np.round(r_curve, 2).tolist()); print("对照 r(K):", np.round(r_rand, 2).tolist())
# 解读（参考）：真实连接组的 mu2 低于保度随机化对照（约 0.058 对 0.070），特征值比更大（约 89 对 57），
# 这可能与半球间连接少形成的瓶颈有关；按主稳定函数判据，相同振荡器的完全同步更难稳定。
# 在本次模拟中，对照网络在较大 K 时 r 更高。两种 K_c 平均场估计相差不小，只作参考。

In [ ]:
# 核对（不用修改）
assert mu2_brain is not None and ratio_brain is not None, "mu2_brain 或 ratio_brain 还没有赋值"
assert 0 < mu2_brain < 10, "mu2_brain 应为正数（网络连通）；若为 0，可能取了最小特征值，或忘了对称化 W"
assert ratio_brain >= 1, "特征值比 mu_N/mu2 不应小于 1"
assert Ks6 is not None and r_curve is not None, "Ks6 或 r_curve 还没有赋值"
Ks6_a, r_a = np.asarray(Ks6, float), np.asarray(r_curve, float)
assert Ks6_a.ndim == 1 and len(Ks6_a) >= 3, "Ks6 应至少包含 3 个耦合强度"
assert r_a.shape == Ks6_a.shape, "r_curve 应与 Ks6 等长"
assert np.all((r_a >= 0) & (r_a <= 1)), "序参量 r 应在 0 到 1 之间"
print("✓ 任务 6 通过")